In [180]:
import torch
from torchvision import datasets,transforms
from torch.utils.data import DataLoader
from torch.optim import Adam
from torch.nn import CrossEntropyLoss
import torch.nn as nn
import numpy as np
from tqdm import tqdm,trange


In [181]:
##needs to be modified for the rectangular issue and for the speed
def patchify(images,n_patches):
    n,c,h,w=images.shape
    assert h==w
    patches=torch.zeros(n,n_patches**2,h*w*c//(n_patches**2))
    patch_size=h//n_patches
    for idx,image in enumerate(images):
        for i in range(n_patches):
            for j in range(n_patches):
                patch=image[:,i*patch_size:(i+1)*patch_size,j*patch_size:(j+1)*patch_size]
                patches[idx,i*n_patches+j]=patch.flatten()
    return patches
    

In [182]:
#same thing here tooo slow 
def get_positional_embedding(sequence_length,d):
    results=torch.ones(sequence_length,d)
    for i in range(sequence_length):
        for j in range(d): 
            results[i][j]=np.sin(i/(10000)**(j/d)) if j%2==0 else np.cos(i/(10000)**(j-1/d))

    return results

In [183]:
class MyMsa(nn.Module):
    def __init__(self,d,n_heads=2):
        super(MyMsa,self).__init__()
        self.d=d
        self.n_heads=n_heads

        assert d%n_heads==0

        d_head=int(d/n_heads)
        self.q_mappings=nn.ModuleList([nn.Linear(d_head,d_head) for _ in range(self.n_heads)])
        self.k_mappings=nn.ModuleList([nn.Linear(d_head,d_head) for _ in range(self.n_heads)])
        self.v_mappings=nn.ModuleList([nn.Linear(d_head,d_head) for _ in range(self.n_heads)])
        self.d_head=d_head
        self.softmax=nn.Softmax(dim=-1)
        self.out_proj=nn.Linear(d,d)

    def forward(self,sequences):
        result=[]
        for sequence in sequences:
            seq_result=[]
            for head in range(self.n_heads):
                q_mapping=self.q_mappings[head]
                k_mapping=self.k_mappings[head]
                v_mapping=self.v_mappings[head]

                seq=sequence[:,head*self.d_head:(head+1)*self.d_head]
                q,k,v=q_mapping(seq),k_mapping(seq),v_mapping(seq)

                attention=self.softmax(q@k.T/(self.d_head**2))
                seq_result.append(attention@v)

            result.append(torch.hstack(seq_result))

        return self.out_proj(torch.cat([torch.unsqueeze(r,dim=0) for r in result]))
                
        

In [184]:
class MyViTBlock(nn.Module):
    def __init__(self, hidden_d, n_heads, mlp_ratio=4):
        super(MyViTBlock, self).__init__()
        self.hidden_d = hidden_d
        self.n_heads = n_heads

        self.norm1 = nn.LayerNorm(hidden_d)
        self.mhsa = MyMsa(hidden_d, n_heads)
        self.norm2 = nn.LayerNorm(hidden_d)
        self.mlp = nn.Sequential(
            nn.Linear(hidden_d, mlp_ratio * hidden_d),
            nn.GELU(),
            nn.Linear(mlp_ratio * hidden_d, hidden_d)
        )

    def forward(self, x):
        out = x + self.mhsa(self.norm1(x))
        out = out + self.mlp(self.norm2(out))
        return out

In [185]:
class MyVit(nn.Module):
    def __init__(self, chw=(1,28,28) , n_patches=7,hidden_d=8,n_blocks=2,n_heads=2,out_d=10):
        super(MyVit,self).__init__()
        self.chw=chw
        self.n_patches=n_patches
        self.n_blocks=n_blocks
        self.n_heads=n_heads
        self.hidden_d=hidden_d
        self.out_d=out_d
        self.patch_size = (chw[1] / n_patches, chw[2] / n_patches)

        assert chw[1] % n_patches == 0
        assert chw[2] % n_patches == 0 
        
        #stage 1  linear map to fit in the transformer dimesnion (changes the dimesnsion to fit the dimesnion of transformer)
        self.input_d= int(chw[0]*self.patch_size[0]*self.patch_size[1])      
        self.linear_mapper=nn.Linear(self.input_d,self.hidden_d)

        # adding class token
        self.class_token=nn.Parameter(torch.rand(1,self.hidden_d))

        #2nd stage positional encoding
        self.pos_embedding=nn.Parameter(torch.tensor(get_positional_embedding(self.n_patches**2+1,self.hidden_d).detach().clone()))

        #3rd transformer block
        self.blocks=nn.ModuleList([MyViTBlock(hidden_d,n_heads) for _ in range(n_blocks)])

        #4 classification mlp
        self.mlp=nn.Sequential(
            nn.Linear(self.hidden_d,self.out_d),
            nn.Softmax(dim=-1)
        )
    
    def forward(self,images):
        n,c,h,w=images.shape
        patches=patchify(images,self.n_patches).to("cuda")
        tokens=self.linear_mapper(patches)
        tokens=torch.stack([torch.vstack((self.class_token,tokens[i]))for i in range(len(tokens))])
        pos_embed=self.pos_embedding.repeat(n,1,1)
        out=tokens+pos_embed

        for block in self.blocks:
            out=block(out)
        out=out[:,0,:]
        return self.mlp(out)

In [186]:
def main():
    transform=transforms.ToTensor()
    train_data=datasets.mnist.MNIST(root="./../datasets",train=True,download=True,transform=transform)
    test_data=datasets.mnist.MNIST(root="./../datasets",train=False,download=True,transform=transform)

    train_loader=DataLoader(train_data,shuffle=True,batch_size=128)
    test_loader=DataLoader(test_data,shuffle=False,batch_size=128)
    device=torch.device("cuda")
    print(f"using device {device}")

    model=MyVit((1,28,28),n_patches=7,n_blocks=2,hidden_d=8,n_heads=2,out_d=10).to(device)
    n_epochs=5
    lr=0.005

    optimizer=Adam(model.parameters(),lr=lr)
    criterion=CrossEntropyLoss()

    for epoch in trange(n_epochs,desc="training"):
        train_loss=0.0
        for batch in train_loader:
            x,y=batch
            x,y=x.to(device),y.to(device)
            y_hat=model(x)
            loss=criterion(y_hat,y)
            train_loss+=loss.detach().cpu().item() /len(train_loader)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        print(f"epoch {epoch+1}/{n_epochs} loss:{train_loss:.2f}")

    with torch.no_grad():
        correct,total=0,0
        test_loss=0.0
        for batch in tqdm(test_loader,desc="testing"):
            x,y=batch
            x,y=x.to(device),y.to(device)
            y_hat=model(x)
            loss=criterion(y_hat,y)
            test_loss+=loss.detach().cpu().item() /len(test_loader)
            correct+=torch.sum(torch.argmax(y_hat,dim=1)==y).detach().cpu().item()
            total+=len(x)

        print(f"test loss :{test_loss:.2f}")
        print(f"test accuracy: {correct/total *100:.2f}")

    
            

main()

/tmp/ipykernel_55/658979365.py:23: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.pos_embedding=nn.Parameter(torch.tensor(get_positional_embedding(self.n_patches**2+1,self.hidden_d).detach().clone()))


using device cuda


training:  20%|██        | 1/5 [05:27<21:51, 327.83s/it]

epoch 1/5 loss:2.14


training:  40%|████      | 2/5 [10:56<16:24, 328.06s/it]

...

epoch 4/5 loss:1.85


training: 100%|██████████| 5/5 [27:22<00:00, 328.44s/it]


epoch 5/5 loss:1.79


testing: 100%|██████████| 79/79 [00:30<00:00,  2.60it/s]

test loss :1.74
test accuracy: 71.58


In [ ]:
https://www.kaggle.com/code/abdelrhmaneldenary/vit-1st/edit